# Preprocessing

In questo notebook verranno eseguite le attivita' di preprocessing del dataset.
Ogni split preprocessato viene salvato nella cartella apposita `data/processed`.

## Download dataset

Scarichiamo il dataset con il codice fornito direttamente da *kaggle*

In [ ]:
from time import perf_counter
import kagglehub

start_time = perf_counter()
print("[Download dataset] Avvio download/caricamento dataset da Kaggle...", flush=True)

# Download diretta da kaggle. Se il dataset e' gia' in cache, KaggleHub restituisce subito il path.
path = kagglehub.dataset_download("msambare/fer2013")

elapsed = perf_counter() - start_time
print(f"[Download dataset] Completato in {elapsed:.1f}s", flush=True)
print("Path to dataset files:", path)


## Visualizzazione immagini

Visualizziamo 12 immagini assieme alle rispettive etichette per farci un'idea delle immagini che compongono il dataset 

In [ ]:
from time import perf_counter
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

start_time = perf_counter()
print("[Visualizzazione] Cerco immagini di esempio...", flush=True)

dataset_path = Path(path)

valid_extensions = [".jpg"]
all_images = [
    p for p in dataset_path.rglob("*") if p.suffix.lower() in valid_extensions
]

print(f"[Visualizzazione] Immagini trovate: {len(all_images)}", flush=True)

if not all_images:
    print(
        f"Nessuna immagine presente nel percorso: {dataset_path}"
    )
else:
    num_samples = 12
    samples = random.sample(all_images, min(num_samples, len(all_images)))

    num_rows = 2
    num_cols = 6

    plt.figure(figsize=(18, 7))

    for idx, img_path in enumerate(samples):
        emotion_class = img_path.parent.name
        title = f"Classe: {emotion_class}"

        img = Image.open(img_path)

        plt.subplot(num_rows, num_cols, idx + 1)

        plt.imshow(img, cmap="gray" if img.mode == "L" else None)
        plt.title(title, fontsize=11, fontweight="bold")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

elapsed = perf_counter() - start_time
print(f"[Visualizzazione] Completata in {elapsed:.1f}s", flush=True)


## Preprocessing

Utilizziamo una funzione per applicare delle leggere modifice alle immagini per generare più campioni.
Le tecniche di preprocessing che andremo ad applicare sono:
- Flip orizzontale
- Traslazione
- Rotazione
- Zoom
- Normalizzazione 

### Funzione per il preprocessing delle immagini

In [ ]:
import torchvision.transforms as transforms

# Pipeline per preprocessing
def get_preprocessing_pipeline():
    preprocessing = transforms.Compose(
        [
            # 1. Flip Orizzontale casuale
            transforms.RandomHorizontalFlip(p=0.5),
            
            # 2. Rotazione casuale
            transforms.RandomRotation(degrees=15, fill=0),
            
            # 3. Traslazione e Zoom leggero
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), fill=0),
            
            # Conversione in tensore
            transforms.ToTensor(),
            
            # 5. Normalizzazione
            #transforms.Normalize(mean=[0.5], std=[0.5])
        ]
    )
    return preprocessing

In [ ]:
# Utilizziamo funzioni di supporto per percorsi e monitoraggio.
# Cosi' si evitano problemi se il notebook viene lanciato dalla root o da notebooks.

import os
from pathlib import Path
from time import perf_counter
from datetime import datetime
from contextlib import contextmanager


def get_wd():
    current_dir = Path(os.getcwd())

    project_root = current_dir if (current_dir / "data").exists() else current_dir.parent

    final_path = project_root / "data"

    return Path(final_path)


def log_step(message):
    now = datetime.now().strftime("%H:%M:%S")
    print(f"[{now}] {message}", flush=True)


@contextmanager
def timed_step(name):
    start_time = perf_counter()
    log_step(f"INIZIO - {name}")
    try:
        yield
    finally:
        elapsed = perf_counter() - start_time
        log_step(f"FINE - {name} ({elapsed:.1f}s)")


def count_images_by_class(root_dir, extension=".jpg"):
    root_dir = Path(root_dir)
    counts = {}
    if not root_dir.exists():
        return counts

    for class_dir in sorted([p for p in root_dir.iterdir() if p.is_dir()]):
        counts[class_dir.name] = len(list(class_dir.glob(f"*{extension}")))

    return counts


# print(get_wd())


### Preprocessing

In [ ]:
import random
import shutil
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms
from tqdm.auto import tqdm

with timed_step("Preprocessing completo"):
    data_root = get_wd()

    dataset_originale_path = data_root / "original" / "train"
    processed_root = data_root / "processed"
    train_processed_path = processed_root / "train"
    validation_processed_path = processed_root / "validation"

    SEED = 42
    VALIDATION_SPLIT = 0.2
    AUGMENTATION_RATIO = 0.40

    random.seed(SEED)

    log_step(f"Dataset originale training: {dataset_originale_path}")
    log_step(f"Dataset preprocessato training: {train_processed_path}")
    log_step(f"Dataset preprocessato validation: {validation_processed_path}")
    log_step(f"Validation split: {VALIDATION_SPLIT:.0%}")
    log_step(f"Augmentation ratio sul training: {AUGMENTATION_RATIO:.0%}")

    pipeline = get_preprocessing_pipeline()

    # Conversione da tensore a immagine normale.
    # Non applichiamo denormalizzazione perche' Normalize e' disattivata nella pipeline.
    def tensor_to_saved_image(tensor):
        tensor = tensor.clamp(0, 1)
        reverse_transform = transforms.ToPILImage()
        return reverse_transform(tensor)

    with timed_step("Lettura classi originali"):
        class_names = sorted([p.name for p in dataset_originale_path.iterdir() if p.is_dir()])

        if not class_names:
            raise FileNotFoundError(f"Nessuna classe trovata in: {dataset_originale_path}")

        original_counts = count_images_by_class(dataset_originale_path)
        for class_name, count in original_counts.items():
            log_step(f"Classe originale {class_name}: {count} immagini")

    if processed_root.exists():
        with timed_step("Pulizia cartella data/processed esistente"):
            log_step(f"Rimuovo: {processed_root}")
            shutil.rmtree(processed_root)

    training_images = []
    validation_images = []

    with timed_step("Creazione split train/validation"):
        for class_name in tqdm(class_names, desc="Split classi"):
            class_dir = dataset_originale_path / class_name
            class_images = sorted([p for p in class_dir.glob("*.jpg")])

            if not class_images:
                log_step(f"Classe senza immagini, salto: {class_name}")
                continue

            random.shuffle(class_images)
            validation_count = int(len(class_images) * VALIDATION_SPLIT)

            if len(class_images) > 1:
                validation_count = max(1, validation_count)

            validation_split_images = class_images[:validation_count]
            training_split_images = class_images[validation_count:]

            training_images.extend(training_split_images)
            validation_images.extend(validation_split_images)

            log_step(
                f"{class_name}: train={len(training_split_images)}, "
                f"validation={len(validation_split_images)}"
            )

    def copy_images(images, source_root, target_root, description):
        for img_path in tqdm(images, desc=description, unit="img"):
            relative_path = img_path.relative_to(source_root)
            target_path = target_root / relative_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, target_path)

    with timed_step("Copia immagini originali nel training preprocessato"):
        log_step(f"Copio {len(training_images)} immagini in data/processed/train")
        copy_images(training_images, dataset_originale_path, train_processed_path, "Copia train")

    with timed_step("Copia immagini originali nella validation preprocessata"):
        log_step(f"Copio {len(validation_images)} immagini in data/processed/validation")
        copy_images(validation_images, dataset_originale_path, validation_processed_path, "Copia validation")

    with timed_step("Generazione immagini augmentate"):
        # Aggiungiamo immagini augmentate solo al training set preprocessato.
        num_immagini_da_preprocessare = int(len(training_images) * AUGMENTATION_RATIO)
        immagini_selezionate = random.sample(training_images, num_immagini_da_preprocessare)

        log_step(f"Genero {num_immagini_da_preprocessare} immagini augmentate nel training set")

        errors = 0
        for img_path in tqdm(immagini_selezionate, desc="Augmentation train", unit="img"):
            relative_path = img_path.relative_to(dataset_originale_path)
            target_path = train_processed_path / relative_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            target_path = target_path.with_name(f"{img_path.stem}_processed{img_path.suffix}")

            try:
                img_originale = Image.open(img_path)
                img_tensor = pipeline(img_originale)
                img_da_salvare = tensor_to_saved_image(img_tensor)
                img_da_salvare.save(target_path)
            except Exception as e:
                errors += 1
                log_step(f"Errore su {img_path}: {e}")

        log_step(f"Augmentation completata con {errors} errori")

    with timed_step("Conteggio finale immagini preprocessate"):
        train_counts = count_images_by_class(train_processed_path)
        validation_counts = count_images_by_class(validation_processed_path)

        log_step(f"Totale training preprocessato: {sum(train_counts.values())} immagini")
        log_step(f"Totale validation preprocessata: {sum(validation_counts.values())} immagini")

        for class_name in sorted(set(train_counts) | set(validation_counts)):
            log_step(
                f"{class_name}: train={train_counts.get(class_name, 0)}, "
                f"validation={validation_counts.get(class_name, 0)}"
            )

    log_step("Preprocessing completato")
    log_step(f"Training output: {train_processed_path}")
    log_step(f"Validation output: {validation_processed_path}")


### Split del dataset

##### Verifica degli split preprocessati

Il preprocessing crea `data/processed/train` e `data/processed/validation`.
Il test set resta `data/original/test`, cosi' la valutazione finale rimane separata e non augmentata.

In [ ]:
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator

with timed_step("Verifica generatori dagli split preprocessati"):
    current_dir = Path.cwd()
    project_root = current_dir if (current_dir / "data").exists() else current_dir.parent

    train_dir = project_root / "data" / "processed" / "train"
    validation_dir = project_root / "data" / "processed" / "validation"
    test_dir = project_root / "data" / "original" / "test"

    IMG_HEIGHT = 48
    IMG_WIDTH = 48
    BATCH_SIZE = 32
    SEED = 42

    if not train_dir.exists():
        raise FileNotFoundError(f"Cartella training preprocessata non trovata: {train_dir}")
    if not validation_dir.exists():
        raise FileNotFoundError(f"Cartella validation preprocessata non trovata: {validation_dir}")
    if not test_dir.exists():
        raise FileNotFoundError(f"Cartella test non trovata: {test_dir}")

    datagen = ImageDataGenerator(rescale=1./255)

    log_step("Creo train_generator")
    train_generator = datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
        class_mode="categorical",
        seed=SEED
    )

    log_step("Creo validation_generator")
    validation_generator = datagen.flow_from_directory(
        validation_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
        class_mode="categorical",
        seed=SEED,
        shuffle=False
    )

    log_step("Creo test_generator")
    test_generator = datagen.flow_from_directory(
        test_dir,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
        class_mode="categorical",
        shuffle=False
    )

    log_step("Split preprocessati caricati correttamente")
    log_step(f"Training samples: {train_generator.samples}")
    log_step(f"Validation samples: {validation_generator.samples}")
    log_step(f"Test samples: {test_generator.samples}")
    log_step(f"Classi: {train_generator.class_indices}")
